<a href="https://colab.research.google.com/github/Abdu1l-0/AIDC-W3-LABS/blob/W3D5/W3D5_extra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json

try:
    levels = json.load(open("bench_report.json"))["runs"][-1]["levels"]
except FileNotFoundError:
    levels = [
        {"concurrency": 1,  "tokens_per_s": 38.2,  "latency_p95_s": 0.9,  "errors": 0},
        {"concurrency": 2,  "tokens_per_s": 71.5,  "latency_p95_s": 1.1,  "errors": 0},
        {"concurrency": 4,  "tokens_per_s": 128.4, "latency_p95_s": 1.4,  "errors": 0},
        {"concurrency": 8,  "tokens_per_s": 210.7, "latency_p95_s": 2.3,  "errors": 0},
        {"concurrency": 16, "tokens_per_s": 224.9, "latency_p95_s": 5.8,  "errors": 0},
    ]
    print("using the sample bench_report -- swap in your own file for a real answer")

for L in levels:
    print(L)

{'concurrency': 1, 'tokens_per_s': 56.21, 'ttft_p50_s': 0.0589, 'ttft_p95_s': 0.0775, 'latency_p95_s': 2.5056, 'errors': 0, 'ok': 20, 'wall_s': 38.571}
{'concurrency': 2, 'tokens_per_s': 105.51, 'ttft_p50_s': 0.0583, 'ttft_p95_s': 0.0766, 'latency_p95_s': 2.3159, 'errors': 0, 'ok': 20, 'wall_s': 20.548}
{'concurrency': 4, 'tokens_per_s': 195.9, 'ttft_p50_s': 0.0594, 'ttft_p95_s': 0.0865, 'latency_p95_s': 2.4194, 'errors': 0, 'ok': 20, 'wall_s': 11.067}
{'concurrency': 8, 'tokens_per_s': 355.65, 'ttft_p50_s': 0.0915, 'ttft_p95_s': 0.1296, 'latency_p95_s': 2.5164, 'errors': 0, 'ok': 20, 'wall_s': 6.096}
{'concurrency': 16, 'tokens_per_s': 533.5, 'ttft_p50_s': 0.2301, 'ttft_p95_s': 0.2391, 'latency_p95_s': 2.996, 'errors': 0, 'ok': 20, 'wall_s': 4.056}


In [ ]:
def cost_per_million_tokens(tokens_per_s, gpu_hourly_usd):
    tokens_per_hour = tokens_per_s * 3600
    million_tokens_per_hour = tokens_per_hour / 1_000_000
    return round(gpu_hourly_usd / million_tokens_per_hour, 4)

GPU_HOURLY_USD = 0.35   # a representative on-demand T4-class price; swap in your real rate

for L in levels:
    L["cost_per_million_tokens_usd"] = cost_per_million_tokens(L["tokens_per_s"], GPU_HOURLY_USD)

for L in levels:
    print(f"c={L['concurrency']:>2}  tok/s={L['tokens_per_s']:>7.1f}  "
          f"p95={L['latency_p95_s']:.2f}s  $/M tok=${L['cost_per_million_tokens_usd']}")

c= 1  tok/s=   56.2  p95=2.51s  $/M tok=$1.7296
c= 2  tok/s=  105.5  p95=2.32s  $/M tok=$0.9215
c= 4  tok/s=  195.9  p95=2.42s  $/M tok=$0.4963
c= 8  tok/s=  355.6  p95=2.52s  $/M tok=$0.2734
c=16  tok/s=  533.5  p95=3.00s  $/M tok=$0.1822


In [ ]:
TARGET_P95_S = 3.0   # your SLO from this afternoon's prediction card

under_target = [L for L in levels if L["latency_p95_s"] <= TARGET_P95_S]
knee = max(under_target, key=lambda L: L["concurrency"]) if under_target else None
print("knee:", knee)

past_knee = [L for L in levels if knee and L["concurrency"] > knee["concurrency"]]
if past_knee:
    cheapest_past_knee = min(past_knee, key=lambda L: L["cost_per_million_tokens_usd"])
    print("cheapest $/M token level past the knee (SLO-violating):", cheapest_past_knee)
    print("-> cheaper on paper, but its p95 already exceeds your SLO -- "
          "not real usable capacity at your target.")

knee: {'concurrency': 16, 'tokens_per_s': 533.5, 'ttft_p50_s': 0.2301, 'ttft_p95_s': 0.2391, 'latency_p95_s': 2.996, 'errors': 0, 'ok': 20, 'wall_s': 4.056, 'cost_per_million_tokens_usd': 0.1822}


In [ ]:
import math

def replicas_needed(required_tokens_per_s, knee_tokens_per_s):
    return math.ceil(required_tokens_per_s / knee_tokens_per_s)

def scale_out_cost(required_tokens_per_s, knee, gpu_hourly_usd):
    n = replicas_needed(required_tokens_per_s, knee["tokens_per_s"])
    return {
        "required_tokens_per_s": required_tokens_per_s,
        "replicas_needed": n,
        "total_hourly_cost_usd": round(n * gpu_hourly_usd, 2),
        "effective_p95_s": knee["latency_p95_s"],   # every replica runs at the same safe knee
    }

targets = [knee["tokens_per_s"] * m for m in (1.0, 1.5, 2.0, 3.0)]
scale_plan = [scale_out_cost(t, knee, GPU_HOURLY_USD) for t in targets]
for row in scale_plan:
    print(row)

{'required_tokens_per_s': 533.5, 'replicas_needed': 1, 'total_hourly_cost_usd': 0.35, 'effective_p95_s': 2.996}
{'required_tokens_per_s': 800.25, 'replicas_needed': 2, 'total_hourly_cost_usd': 0.7, 'effective_p95_s': 2.996}
{'required_tokens_per_s': 1067.0, 'replicas_needed': 2, 'total_hourly_cost_usd': 0.7, 'effective_p95_s': 2.996}
{'required_tokens_per_s': 1600.5, 'replicas_needed': 3, 'total_hourly_cost_usd': 1.05, 'effective_p95_s': 2.996}


In [ ]:
report = {
    "gpu_hourly_usd": GPU_HOURLY_USD,
    "target_p95_s": TARGET_P95_S,
    "levels": levels,
    "knee": knee,
    "scale_out_plan": scale_plan,
}
with open("cost_report.json", "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))

{
  "gpu_hourly_usd": 0.35,
  "target_p95_s": 3.0,
  "levels": [
    {
      "concurrency": 1,
      "tokens_per_s": 56.21,
      "ttft_p50_s": 0.0589,
      "ttft_p95_s": 0.0775,
      "latency_p95_s": 2.5056,
      "errors": 0,
      "ok": 20,
      "wall_s": 38.571,
      "cost_per_million_tokens_usd": 1.7296
    },
    {
      "concurrency": 2,
      "tokens_per_s": 105.51,
      "ttft_p50_s": 0.0583,
      "ttft_p95_s": 0.0766,
      "latency_p95_s": 2.3159,
      "errors": 0,
      "ok": 20,
      "wall_s": 20.548,
      "cost_per_million_tokens_usd": 0.9215
    },
    {
      "concurrency": 4,
      "tokens_per_s": 195.9,
      "ttft_p50_s": 0.0594,
      "ttft_p95_s": 0.0865,
      "latency_p95_s": 2.4194,
      "errors": 0,
      "ok": 20,
      "wall_s": 11.067,
      "cost_per_million_tokens_usd": 0.4963
    },
    {
      "concurrency": 8,
      "tokens_per_s": 355.65,
      "ttft_p50_s": 0.0915,
      "ttft_p95_s": 0.1296,
      "latency_p95_s": 2.5164,
      "errors": 0,


In [ ]:
#!/usr/bin/env python3
# Green check for the extra W3D5 lab (cost per million tokens, scale-out).
# Run next to cost_report.json:  python verify.py
# Prints exactly one line last: GREEN CHECK: PASS  or  GREEN CHECK: FAIL (<reason>)
# stdlib only.
#
# The lab is pure arithmetic over the student's own bench levels, so this
# recomputes EVERYTHING from the levels in the report: per-level cost, knee
# selection, and the whole scale-out plan. It works identically for the sample
# bench and a student's real one.
import json, math, os
from typing import NoReturn


class _Stop(Exception):
    pass


def _fail(reason) -> NoReturn:
    print("GREEN CHECK: FAIL (%s)" % reason)
    raise _Stop()


def main():
    if not os.path.isfile("cost_report.json"):
        _fail("cost_report.json not found; run Step 5 first")
    try:
        with open("cost_report.json") as f:
            r = json.load(f)
    except json.JSONDecodeError as e:
        _fail("cost_report.json is not valid JSON: %s" % e)

    for key in ("gpu_hourly_usd", "target_p95_s", "levels", "knee", "scale_out_plan"):
        if key not in r:
            _fail("missing key '%s'" % key)
    rate, slo = r["gpu_hourly_usd"], r["target_p95_s"]
    if not isinstance(rate, (int, float)) or rate <= 0:
        _fail("gpu_hourly_usd must be a positive dollars-per-hour figure")
    if not isinstance(slo, (int, float)) or slo <= 0:
        _fail("target_p95_s must be a positive SLO in seconds")

    levels = r["levels"]
    if not isinstance(levels, list) or len(levels) < 3:
        _fail("levels must hold the bench sweep (at least 3 concurrency levels)")
    for L in levels:
        for f_ in ("concurrency", "tokens_per_s", "latency_p95_s",
                   "cost_per_million_tokens_usd"):
            if not isinstance(L.get(f_), (int, float)):
                _fail("level %r lacks numeric %s" % (L.get("concurrency"), f_))
        if not L["tokens_per_s"] or L["tokens_per_s"] <= 0:
            _fail("level %s reports tokens_per_s <= 0 (an all-error level); rerun the sweep" % L.get("concurrency"))
        want_cost = round(rate / (L["tokens_per_s"] * 3600 / 1_000_000), 4)
        if abs(L["cost_per_million_tokens_usd"] - want_cost) > max(0.0002, want_cost * 0.01):
            _fail("concurrency %s: cost %.4f, the formula gives %.4f "
                  "(tokens/s vs tokens/hour, or a non-hourly rate?)"
                  % (L["concurrency"], L["cost_per_million_tokens_usd"], want_cost))

    under = [L for L in levels if L["latency_p95_s"] <= slo]
    if not under:
        _fail("no level sits under the SLO, so no knee exists; the report "
              "should not have gotten this far (see failure modes)")
    want_knee = max(under, key=lambda L: L["concurrency"])
    knee = r["knee"]
    if not isinstance(knee, dict) or knee.get("concurrency") != want_knee["concurrency"]:
        _fail("knee is concurrency %s; the largest level under the %.1fs SLO "
              "is concurrency %s" % ((knee or {}).get("concurrency"), slo,
                                     want_knee["concurrency"]))

    plan = r["scale_out_plan"]
    want_targets = [round(want_knee["tokens_per_s"] * m, 6) for m in (1.0, 1.5, 2.0, 3.0)]
    if not isinstance(plan, list) or len(plan) != 4:
        _fail("scale_out_plan must hold the four multiples 1.0, 1.5, 2.0, 3.0")
    for row, want_req in zip(plan, want_targets):
        req = row.get("required_tokens_per_s")
        if not isinstance(req, (int, float)) or abs(req - want_req) > max(0.5, want_req * 0.01):
            _fail("plan targets must be the knee's throughput x (1, 1.5, 2, 3); "
                  "got %r, expected %.1f" % (req, want_req))
        want_n = math.ceil(want_req / want_knee["tokens_per_s"] - 1e-9)
        if row.get("replicas_needed") != want_n:
            _fail("required %.1f tok/s: replicas_needed=%r, ceil gives %d"
                  % (req, row.get("replicas_needed"), want_n))
        want_cost = round(want_n * rate, 2)
        if abs(row.get("total_hourly_cost_usd", 1e9) - want_cost) > 0.011:
            _fail("required %.1f tok/s: hourly cost %r, %d replicas at %.2f/h "
                  "gives %.2f" % (req, row.get("total_hourly_cost_usd"),
                                  want_n, rate, want_cost))
        if abs(row.get("effective_p95_s", 1e9) - want_knee["latency_p95_s"]) > 0.011:
            _fail("effective_p95_s must stay at the knee's p95: replicas run at "
                  "the safe concurrency, that is the whole model")

    print("recomputed costs, knee and scale-out plan all agree")
    print("GREEN CHECK: PASS")


if __name__ == "__main__":
    try:
        main()
    except _Stop:
        raise SystemExit(1)


recomputed costs, knee and scale-out plan all agree
GREEN CHECK: PASS
